In [3]:
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# ---- Dummy corpus ----
sentences = [
    "I love machine learning",
    "This model learns representations",
    "BERT uses attention mechanisms",
    "Transformers process sequences",
    "Deep learning is powerful",
    "I like natural language processing"
]

# special tokens
PAD = "[PAD]"; UNK = "[UNK]"; CLS = "[CLS]"; SEP = "[SEP]"; MASK = "[MASK]"
special_tokens = [PAD, UNK, CLS, SEP, MASK]

# build vocab (very small)
tokens = set()
for s in sentences:
    tokens.update(s.lower().split())
vocab_list = special_tokens + sorted(tokens)
token2id = {t:i for i,t in enumerate(vocab_list)}
id2token = {i:t for t,i in token2id.items()}

vocab_size = len(vocab_list)
print("vocab_size =", vocab_size,vocab_list)


vocab_size = 27 ['[PAD]', '[UNK]', '[CLS]', '[SEP]', '[MASK]', 'attention', 'bert', 'deep', 'i', 'is', 'language', 'learning', 'learns', 'like', 'love', 'machine', 'mechanisms', 'model', 'natural', 'powerful', 'process', 'processing', 'representations', 'sequences', 'this', 'transformers', 'uses']


In [12]:
MAX_LEN = 12  # small for demo
MASK_PROB = 0.15

def tokenize_text(text):
    return text.lower().split()

def make_example(sent_a, sent_b, is_next):
    # Build token sequence: [CLS] A [SEP] B [SEP]
    a_tokens = tokenize_text(sent_a)
    b_tokens = tokenize_text(sent_b)
    tokens = [CLS] + a_tokens + [SEP] + b_tokens + [SEP]
    token_ids = [token2id.get(t, token2id[UNK]) for t in tokens]
    
    # token_type_ids (segment ids)
    seg_ids = []
    seen_sep = False
    for t in tokens:
        seg_ids.append(0 if not seen_sep else 1)
        if t == SEP:
            seen_sep = True
    
    # attention mask (1 for tokens, 0 for pad)
    attention_mask = [1]*len(token_ids)
    
    # pad to MAX_LEN
    pad_len = MAX_LEN - len(token_ids)
    if pad_len < 0:
        token_ids = token_ids[:MAX_LEN]  # truncate (simple)
        seg_ids = seg_ids[:MAX_LEN]
        attention_mask = attention_mask[:MAX_LEN]
    else:
        token_ids += [token2id[PAD]]*pad_len
        seg_ids += [0]*pad_len
        attention_mask += [0]*pad_len
    
    # Create MLM labels and masked_input_ids
    input_ids = token_ids.copy()
    mlm_labels = [-100] * MAX_LEN  # -100 means ignore
    
    # candidate positions to mask: not special tokens CLS/SEP/PAD
    cand_positions = [i for i,tid in enumerate(input_ids) 
                      if id2token[tid] not in (CLS, SEP, PAD)]
# print('cand_positions',cand_positions)
    # print('len(cand_positions)',len(cand_positions))
    num_to_mask = max(1, int(round(len(cand_positions) * MASK_PROB)))
    # print('num_to_mask',num_to_mask)
    mask_positions = random.sample(cand_positions, num_to_mask)
    # print('mask_positions',mask_positions)
    
    for pos in mask_positions:
        orig_id = input_ids[pos]
        prob = random.random()
        if prob < 0.8:
            input_ids[pos] = token2id[MASK]
        elif prob < 0.9:
            # replace with random token
            input_ids[pos] = random.randrange(vocab_size)
        else:
            # keep original
            pass
        mlm_labels[pos] = orig_id
    
    # print('input_ids',input_ids)
    # print('token_type_ids',seg_ids)
    # print('attention_mask',attention_mask)
    # print('mlm_labels',mlm_labels)  
    # print('nsp_label',np.array(int(is_next), dtype=np.int32))
    
    
    return {
        "input_ids": np.array(input_ids, dtype=np.int32),
        "token_type_ids": np.array(seg_ids, dtype=np.int32),
        "attention_mask": np.array(attention_mask, dtype=np.int32),
        "mlm_labels": np.array(mlm_labels, dtype=np.int32),
        "nsp_label": np.array(int(is_next), dtype=np.int32)
    }

# Build dataset of positive and negative pairs
examples = []
for i in range(len(sentences)-1):
    # positive pair: next sentence
    examples.append(make_example(sentences[i], sentences[i+1], True))
    # negative (random) pair
    a = sentences[i]
    b = random.choice([s for j,s in enumerate(sentences) if j != i+1])
    examples.append(make_example(a, b, False))

# Convert to arrays
def to_array(exs, key):
    return np.stack([e[key] for e in exs], axis=0)

input_ids = to_array(examples, "input_ids")
token_type_ids = to_array(examples, "token_type_ids")
attention_masks = to_array(examples, "attention_mask")
mlm_labels = to_array(examples, "mlm_labels")   # -100 for ignored
nsp_labels = to_array(examples, "nsp_label")    # 0/1

print("input_ids shape:", input_ids.shape)
print("mlm_labels example:\n", mlm_labels.shape)
print("examples[0]",examples[0])


input_ids shape: (10, 12)
mlm_labels example:
 (10, 12)
examples[0] {'input_ids': array([ 2,  8, 14, 15, 11,  3, 24, 22, 12, 22,  3,  0], dtype=int32), 'token_type_ids': array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0], dtype=int32), 'attention_mask': array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], dtype=int32), 'mlm_labels': array([-100, -100, -100, -100, -100, -100, -100,   17, -100, -100, -100,
       -100], dtype=int32), 'nsp_label': array(1, dtype=int32)}


In [14]:
from tensorflow.keras.layers import Lambda

class TransformerBlock(layers.Layer):
    def __init__(self, d_model, num_heads, dff, dropout=0.1):
        super().__init__()
        self.mha = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model//num_heads)
        self.ffn = tf.keras.Sequential([
            layers.Dense(dff, activation="relu"),
            layers.Dense(d_model)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(dropout)
        self.dropout2 = layers.Dropout(dropout)
    def call(self, x, mask=None, training=False):
        attn_out = self.mha(x, x, attention_mask=mask)  # self-attention
        attn_out = self.dropout1(attn_out, training=training)
        out1 = self.layernorm1(x + attn_out)
        ffn_out = self.ffn(out1)
        ffn_out = self.dropout2(ffn_out, training=training)
        out2 = self.layernorm2(out1 + ffn_out)
        return out2

def build_small_bert(vocab_size, seq_len, d_model=32, num_heads=4, dff=64, num_layers=2):
    input_ids = layers.Input(shape=(seq_len,), dtype=tf.int32, name="input_ids")
    token_type_ids = layers.Input(shape=(seq_len,), dtype=tf.int32, name="token_type_ids")
    attention_mask = layers.Input(shape=(seq_len,), dtype=tf.int32, name="attention_mask")
    
    # Embeddings
    token_emb = layers.Embedding(vocab_size, d_model, name="token_emb")(input_ids)
    pos_indices = tf.range(start=0, limit=seq_len, delta=1)
    pos_emb_layer = layers.Embedding(seq_len, d_model, name="pos_emb")
    pos_emb = pos_emb_layer(pos_indices)[tf.newaxis, :, :]   # (1, seq_len, d_model)
    type_emb = layers.Embedding(2, d_model, name="type_emb")(token_type_ids)
    
    x = token_emb + pos_emb + type_emb
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    x = layers.Dropout(0.1)(x)
    
    # attention mask conversion: Keras MultiHeadAttention expects boolean mask with shape (batch, seq_len)
    # bool_mask = tf.cast(attention_mask[:, tf.newaxis, :], dtype=tf.bool)  # (batch, 1, seq_len) for broadcasting

    

# Wrap mask preprocessing inside a Keras layer
    mask_expansion = Lambda(lambda x: tf.cast(x[:, tf.newaxis, :], tf.bool))
    bool_mask = mask_expansion(attention_mask)

    for _ in range(num_layers):
        x = TransformerBlock(d_model, num_heads, dff)(x, mask=bool_mask)
    
    sequence_output = x  # (batch, seq_len, d_model)
    pooled_output = layers.Lambda(lambda y: y[:, 0, :], name="cls_token")(sequence_output)  # CLS
    
    # MLM head: project per-token to vocab scores
    mlm_logits = layers.Dense(vocab_size, name="mlm_logits")(sequence_output)  # (batch, seq_len, vocab_size)
    
    # NSP head: CLS -> logits for binary classification
    nsp_logits = layers.Dense(1, name="nsp_logits")(pooled_output)  # (batch, 1)
    
    model = tf.keras.Model(inputs=[input_ids, token_type_ids, attention_mask],
                           outputs=[mlm_logits, nsp_logits])
    return model

seq_len = MAX_LEN
model = build_small_bert(vocab_size, seq_len)
model.summary()


/home/ankur/Desktop/dl_1000/.venv-1000-ml-dl/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_block' (of type TransformerBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/ankur/Desktop/dl_1000/.venv-1000-ml-dl/lib/python3.12/site-packages/keras/src/layers/layer.py:970: UserWarning: Layer 'transformer_block_1' (of type TransformerBlock) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_ids           │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_emb           │ (None, 12, 32)    │        864 │ input_ids[0][0]   │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_type_ids      │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 12, 32)    │          0 │ token_emb[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ type_emb            │ (None, 12, 32)    │         64 │ token_type_ids[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 12, 32)    │          0 │ add_2[0][0],      │
│                     │                   │            │ type_emb[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 12, 32)    │         64 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_mask      │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 12, 32)    │          0 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda (Lambda)     │ (None, 1, 12)     │          0 │ attention_mask[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block   │ (None, 12, 32)    │      8,544 │ dropout_1[0][0],  │
│ (TransformerBlock)  │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ transformer_block_1 │ (None, 12, 32)    │      8,544 │ transformer_bloc… │
│ (TransformerBlock)  │                   │            │ lambda[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cls_token (Lambda)  │ (None, 32)        │          0 │ transformer_bloc… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mlm_logits (Dense)  │ (None, 12, 27)    │        891 │ transformer_bloc… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ nsp_logits (Dense)  │ (None, 1)         │         33 │ cls_token[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 19,004 (74.23 KB)

 Trainable params: 19,004 (74.23 KB)

 Non-trainable params: 0 (0.00 B)

In [23]:
loss_fn_mlm = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
loss_fn_nsp = tf.keras.losses.BinaryCrossentropy(from_logits=True)

optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)

batch_size = 2
dataset = tf.data.Dataset.from_tensor_slices((
    {"input_ids": input_ids, "token_type_ids": token_type_ids, "attention_mask": attention_masks},
    {"mlm": mlm_labels, "nsp": nsp_labels}
)).shuffle(10).batch(batch_size)

@tf.function
def train_step(batch_inputs, batch_labels):
    with tf.GradientTape() as tape:
        mlm_logits, nsp_logits = model([batch_inputs["input_ids"],
                                       batch_inputs["token_type_ids"],
                                       batch_inputs["attention_mask"]],
                                      training=True)
        # MLM loss (only consider positions where label != -100)
        mlm_label = batch_labels["mlm"]  # shape (b, seq_len)
        mask = tf.not_equal(mlm_label, -100)  # boolean mask
        
        # Flatten & select positions
        mlm_logits_flat = tf.reshape(mlm_logits, [-1, vocab_size])
        mlm_label_flat = tf.reshape(mlm_label, [-1])
        mask_flat = tf.reshape(mask, [-1])
        
        selected_logits = tf.boolean_mask(mlm_logits_flat, mask_flat)
        selected_labels = tf.boolean_mask(mlm_label_flat, mask_flat)
        if tf.shape(selected_labels)[0] > 0:
            mlm_loss_batch = tf.reduce_mean(loss_fn_mlm(selected_labels, selected_logits))
        else:
            mlm_loss_batch = 0.0
        
        # NSP loss
        nsp_label = tf.cast(batch_labels["nsp"], tf.float32)
        nsp_logits_flat = tf.reshape(nsp_logits, [-1, 1])
        nsp_loss_batch = loss_fn_nsp(nsp_label, nsp_logits_flat)
        
        loss = mlm_loss_batch + nsp_loss_batch
    
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss, mlm_loss_batch, nsp_loss_batch

# small training run
EPOCHS = 8
for epoch in range(EPOCHS):
    total_loss = 0.0
    steps = 0
    for batch in dataset:
        batch_inputs, batch_labels = batch
        loss_val, mlm_l, nsp_l = train_step(batch_inputs, batch_labels)
        total_loss += float(loss_val)
        steps += 1
    print(f"Epoch {epoch+1}/{EPOCHS}  loss={total_loss/steps:.4f}")


Epoch 1/8  loss=0.0075
Epoch 2/8  loss=0.0082
Epoch 3/8  loss=0.0085
Epoch 4/8  loss=0.0045
Epoch 5/8  loss=0.0044
Epoch 6/8  loss=0.0042
Epoch 7/8  loss=0.0043
Epoch 8/8  loss=0.0040


In [ ]:
fine tuning

In [ ]:
from tensorflow.keras import Model, layers

# Reuse encoder inside the trained BERT model
input_ids_layer = model.input[0]          # (batch, seq_len)
type_ids_layer = model.input[1]           # (batch, seq_len)
attn_mask_layer = model.input[2]          # (batch, seq_len)

# Get encoder output (sequence hidden states)
sequence_output = model.get_layer("encoder_output").output  
# (batch, seq_len, hidden_size)

# Take [CLS] token (index 0)
cls_rep = sequence_output[:, 0, :]   # shape (batch, hidden_size)

# Classification head
clf_out = layers.Dense(1, activation="sigmoid")(cls_rep)

# Build classification model
clf_model = Model(
    inputs=[input_ids_layer, type_ids_layer, attn_mask_layer],
    outputs=clf_out
)


In [ ]:
clf_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # small LR for fine-tuning
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

# Dummy binary sentiment labels
num_examples = input_ids.shape[0]
sentiment_labels = np.random.randint(0, 2, size=(num_examples, 1))

clf_model.fit(
    {"input_ids": input_ids,
     "token_type_ids": token_type_ids,
     "attention_mask": attention_masks},
    sentiment_labels,
    epochs=3,
    batch_size=2
)


In [ ]:
import math
class Solution:
    def median(self, arr):
        arr.sort()
        arr_len = len(arr)
        
        if(arr_len%2==0):
            left = int(arr_len/2-1)
            right = left+1
            
            return math.floor(( arr[left] + arr[right])/2)
        else :
            median_index = int(arr_len / 2)
            return arr[median_index]
    
    def mean(self, arr):
        arr_len = len(arr)
        return math.floor(sum(arr)/arr_len);